In [ ]:
import math
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np
import random
import pickle
from image_utils import *

In [ ]:
load_dotenv()

WIDTH = int(os.getenv("WIDTH"))
HEIGHT = int(os.getenv("HEIGHT"))
LABELS = os.getenv("LABELS", "").split(",")
FOLDER_NAME = "EnsembleB"
PATH = Path.cwd() / FOLDER_NAME

FOLDER_PRETRAITE_NAME = "EnsembleB_pretraite"
PATH_PRETRAITE = Path.cwd() / FOLDER_PRETRAITE_NAME

x_file_path = Path("X.pickle")
y_file_path = Path("y.pickle")

In [ ]:
def dataset_exists():
    return x_file_path.exists() and y_file_path.exists()

In [ ]:
def process_image(image, thresh, ratio, label, min_area=100):
    rois, labeled_rois = find_rois_2(image, thresh, ratio, min_area)
    #print(len(rois))
    resized_image = resize_ignore_aspect(image, WIDTH, HEIGHT)
    filtered_roi, labeled_filtered = filter_rois(rois, label)
    # TODO Augmenter les images (rotation, flip, etc.) pour augmenter la variété du dataset
    augmented_rois = []
    """Liste des images augmentées et distortionées pour entraîner le modèle avec plus de données"""

    if filtered_roi is not None :
        # 0: L'image originale rapetissée à WIDTH X HEIGHT pixels sans tenir compte des proportions
        base_roi = cv2.resize(filtered_roi, (WIDTH, HEIGHT))
        augmented_rois.append(base_roi)

        # 1, 2, 3: Miroir de l'image
        augmented_rois.append(cv2.flip(base_roi, 0)) # Vertical (0 pour l'axe des x)
        augmented_rois.append(cv2.flip(base_roi, 1)) # Horizontal (1 pour l'axe des y)
        augmented_rois.append(cv2.flip(base_roi, -1)) # Des deux vôté (-1 pour les deux axes)

        # 4, 5: Changement de luminosité (assombrir et éclaircir)
        matrix = np.ones(base_roi.shape, dtype = "uint8") * 50 # Remplir une matrice WIDTH X HEIGHT de cellules égales à 50
        augmented_rois.append(cv2.subtract(base_roi, matrix)) # Assombrir en soustrayant 50 à toutes les cellules de base_roi
        matrix = np.ones(base_roi.shape, dtype = "uint8") * 100 #
        augmented_rois.append(cv2.add(base_roi, matrix)) # Éclaircir en additionnant 100 à toutes les cellules de base_roi

        # 6, 7: Rotations
        centre = (WIDTH/2, HEIGHT/2)
        matrix_plus_45 = cv2.getRotationMatrix2D(centre, 45, 1)
        matrix_moins_45 = cv2.getRotationMatrix2D(centre, -45, 1)
        augmented_rois.append(cv2.warpAffine(base_roi, matrix_plus_45, (WIDTH, HEIGHT))) # Rotation sens anti-horaire
        augmented_rois.append(cv2.warpAffine(base_roi, matrix_moins_45, (WIDTH, HEIGHT))) # Rotation sens horaire

        # Image
    if labeled_rois is not None:
        labeled_rois = resize_ignore_aspect(labeled_rois, WIDTH, HEIGHT)
    if labeled_filtered is not None:
        labeled_filtered = resize_ignore_aspect(labeled_filtered, WIDTH, HEIGHT)
    return augmented_rois, labeled_rois, labeled_filtered

In [ ]:
def dataset_fill(path):
    data_set = []
    counter = 0
    if PATH_PRETRAITE.exists():
        shutil.rmtree(PATH_PRETRAITE)
    #PATH_PRETRAITE.mkdir(parents=True, exist_ok=True)

    for path in Path(path).rglob('*'):
        if path.is_dir() and path.name in LABELS:

            label = path.name
            shapeCategory = label[:-1] # Si label est "Cercle2", on a "Cercle"
            class_num = LABELS.index(label)

            print(f"Label: {label}, Class Number: {class_num}")

            # Créer le sous-dossier de la classe dans l'ensembleB prétraité
            class_dir = PATH_PRETRAITE / shapeCategory / label
            class_dir.mkdir(parents=True, exist_ok=True)

            for file in path.glob('*'):
                if file.is_file():
                    img = load_image(file)
                    #TODO: Revoir le Preprocess, il y a des Faux Negatifs
                    preprocess_img, thresh, ratio = preprocess_for_contours(img)
                    augmented_rois, labeled_rois, labeled_roi = process_image(preprocess_img, thresh, ratio, label)
                    if not augmented_rois:
                        continue
                    # Itérer sur toutes les images générées
                    for idx, aug_roi in enumerate(augmented_rois):
                        # Créer un nom de fichier unique
                        new_filename = f"{idx}_{file.name}"

                        # Sauvegarder l'image dans l'EnsembleB prétraité
                        cv2.imwrite(str(class_dir / new_filename), aug_roi)

                        data_set.append([aug_roi, class_num, label, new_filename])
                        counter +=1
                    # if counter >= 1520:
                    #     counter = 0
                    #     break
    return data_set


In [ ]:
def dataset_write(data_set):
    # features vector
    X = []

    # lables vector
    y = []

    filenames = []

    # Taking features and labels from dataset
    # data_set.append([roi, class_num, label, file.name])
    for features, class_num, label, file_name in data_set:
        X.append([features])
        y.append(label)
        filenames.append(file_name)

    # Converts each image matrix to an image
    X = np.array(X).reshape(-1, WIDTH*HEIGHT)

    # Creating the files containing all the information about your model and saving them to the disk
    print("Saving data to disk...")
    pickle_out = open("X.pickle", "wb")
    pickle.dump(X, pickle_out)
    pickle_out.close()

    print("Saving labels to disk...")
    pickle_out = open("y.pickle", "wb")
    pickle.dump(y, pickle_out)
    pickle_out.close()

    print("Saving filenames to disk...")
    pickle_out = open("filenames.pickle", "wb")
    pickle.dump(filenames, pickle_out)
    pickle_out.close()


In [ ]:
def dataset_creation(path, override = False):
    if dataset_exists() and not override:
        print("Dataset already exists. Skipping creation")
        return

    if not path.exists():
        print("Directory does NOT exist")
        return

    data_set = dataset_fill(path)

    # Shuffles the images
    if len(data_set) > 0:
        random.shuffle(data_set)
        dataset_write(data_set)

In [ ]:
def dataset_load():
    if not dataset_exists():
        print("Dataset does NOT exist. Please create it first.")
        return None, None

    print("Loading dataset from disk...")
    with open("X.pickle", "rb") as f:
        X = pickle.load(f)

    with open("y.pickle", "rb") as f:
        y = pickle.load(f)

    with open("filenames.pickle", "rb") as f:
        filenames = pickle.load(f)

    return X, y, filenames

In [ ]:
def plot_image_grid(X, y, filenames, height, width, num_rows=4, num_cols=4):
    images_per_page = num_rows * num_cols
    total_images = len(X)
    print(total_images)
    total_pages = math.ceil(total_images / images_per_page)
    for page in range(total_pages):
        plt.figure(figsize=(3*num_cols, 3*num_rows))
        start_idx = page * images_per_page
        end_idx = min(start_idx + images_per_page, total_images)
        print(page)
        for i in range(start_idx, end_idx):
            plt.subplot(num_rows, num_cols, i - start_idx + 1)
            plt.grid(False)
            plt.xticks([])
            plt.yticks([])
            plt.imshow(X[i].reshape(height, width), cmap="gray")
            plt.xlabel(f"{y[i]}")
            plt.title(f"{filenames[i]}")

        plt.tight_layout()
        plt.show()  # Affiche chaque “page” séparément

In [ ]:
override = True
"""Si override est False, le dataset ne se recrée pas s'il existe déjà (X.pickle et y.pickle et EnsembleB existent) et s'il est True, il se recrée peu importe. """

In [ ]:
dataset_creation(PATH, override)

In [ ]:
X, y, filenames= dataset_load()

In [ ]:
#plot_image_grid(X, y, filenames, HEIGHT, WIDTH)